# 03 · Feasible region: what the data allow before the first fit

Every observation we have, from Gogoi 2024 and from the raw lab spectra (notebook 04), becomes a constraint on the model parameters. The notebook draws what the constraints leave, for a ladder of models that starts with one barrier, as in the reference model. **Nothing is fitted here**: the first fit starts from this region.

It replaces `03_experiment_plan.ipynb`, which was written before the lab spectra existed. The experiment-design part of that version is not repeated here.

**Model ladder**

| | Model | Free parameters | Fixed by assumption |
|---|---|---|---|
| M0 | one barrier for all nine reactions: capped BEP (`reference_bep`) or Marcus | 1 | every family equally fast at equal ΔG_rxn |
| M1 | one Marcus barrier g per family (`level1` structure) | 4 | R1–R3 share one g; ΔG_rxn as computed |
| M1-split | M1 with one g per hydrolysis step | 6 | ΔG_rxn as computed |
| M3 | M1 with ΔG_rxn of R2 and R3 free within their standard error | 6 | not coded: tested from stoichiometry (§5.3) |
| M4 | M1 with part of the water unavailable | 5 | not coded: needs a Karl Fischer titration |

**Rule, fixed before looking at the results:** a model is rejected when no parameter value satisfies every observation. Nothing is ranked here; the fit ranks the models that survive.

**How an observation becomes a constraint.** An observation is a mixture, a temperature, the interval in which the time since mixing lies, and windows on what the spectrum shows (for example "TMSPA holds at least 0.89 of all P"). A model passes if some time in the interval puts every window right. A window that needs little reaction is decided by the earliest possible time; one that needs enough reaction, by the latest.

**Assumptions**
- Thermochemistry and barrier relations as in notebook 01: ωB97M-V G (298 K) + MACE ΔE_solv in EC, ΔS‡ = 0.
- The EC/DEC samples are simulated in EC at 7.1 M (buffered), as one phase, at the temperature stored in each file.
- Lab amounts come from ³¹P (room temperature, NOE off) and from the Si–CH₃ region of ¹³C. ¹H is not used (§2.1).
- **Times.** The files give a lower bound on the time since mixing: the sample existed before its first spectrum. The upper bound is not recorded. It is set by `ASSUMED_MAX_AGE_H` below, and §4.2 shows what changes with it.

**NMR data.** The spectra are read from the folder given by the environment variable `LAB_NMR_DIR` (see `.env.example`). The per-spectrum tables this notebook exports go to the data folder (`ATOM_DATA_DIR`).

**Source:** Y. Alcaraz Galván; outputs computed from the Tank dataset snapshot (P. Broqvist) and NMR data from experiments; Gogoi et al., J. Phys. Chem. C 2024, 128, 1654

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the repository importable whether the notebook runs from notebooks/ or from the repository root
REPO = Path.cwd() if (Path.cwd() / 'kinetics').is_dir() else Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
from IPython.display import display

from kinetics import calculate_network_thermo, get_model
from kinetics.constants import ZERO_CELSIUS_K
from kinetics.data import (
    DEFAULT_LAB_NMR_DIR, build_lab_nmr_inventory, find_heated_windows, load_lab_sample_folders, tabulate_area_shares,
)
from kinetics.microkinetics import tabulate_family_barriers
from kinetics.reactor import calculate_water_series_c0, simulate_phosphate_path
from kinetics.thermo import calculate_solvation_covariance
from kinetics.fitting import (
    FAMILIES, SHARED_BARRIER, build_observation, build_share_window, calculate_equilibrium_locus,
    calculate_path_distance, calculate_required_water_M, calculate_rt_equivalent_hours, describe_observations,
    find_allowed_intervals, find_closest_on_locus, intersect_intervals, observations_from_controls,
    replace_observation, select_observations, split_family_by_reaction, summarize_region, trace_boundaries,
)
from demo import fitting_plots, style, tables

style.apply_style()
pd.set_option('display.max_colwidth', None)

ASSUMED_MAX_AGE_H = 24.0   # ASSUMED: longest time from mixing to a sample's first spectrum (not in the files)
K_SIGMA = 2.0              # windows on measured shares are ± 2 standard errors

## 1. The model and what is free

The computed thermochemistry (notebook 01, §3) and the barriers each model starts from. When one barrier is scanned, the others keep their `level1` values.

In [ ]:
thermo = calculate_network_thermo(298.15)
display(tables.format_thermo_table(thermo))

REFERENCE = get_model('level1')     # barriers the unscanned families keep
SPLIT, SPLIT_NETWORK, HYDROLYSIS_STEPS = split_family_by_reaction(REFERENCE, 'hydrolysis')   # M1-split
tables.format_family_barriers(tabulate_family_barriers(['reference_bep', 'level1']))

**Reading.** Every intrinsic barrier is free. The computed ΔG_rxn of R2–R4 and R6–R8 is smaller than its standard error, so their sign is not resolved; §5.3 uses this.

## 2. Observations

### 2.1 Measured shares in the lab spectra

- **³¹P:** every room-temperature spectrum of a TMSPa sample with the NOE off and a relaxation delay of at least 5 s, the settings that all quantitative ³¹P spectra share (notebook 04, §2).
- **¹³C:** the Si–CH₃ region of the two TMSOH samples. Each Si carries three methyl carbons, so area shares are Si shares; the NOE is assumed equal for the three methyls. A small unassigned line at −1.75 ppm falls in the TMSOEG window and is counted with it.
- **Uncertainty:** each share is computed with three baselines (`calculate_area_shares`). Its standard error combines the white noise and half the spread between the baselines.

In [ ]:
inventory = build_lab_nmr_inventory(DEFAULT_LAB_NMR_DIR).drop_duplicates('data_md5')
SAMPLES = load_lab_sample_folders()['raw_sample_folders']   # sample → folders holding its spectra (notebook 04, §1)
inventory['sample'] = inventory['folder'].map({f: s for s, folders in SAMPLES.items() for f in folders})
inventory = inventory[inventory['sample'].notna()].copy()
first_spectrum = inventory.groupby('sample')['acquired_at'].transform('min')
inventory['hours_since_first'] = (inventory['acquired_at'] - first_spectrum).dt.total_seconds() / 3600.0

P_WINDOWS_PPM = {'TMSPA': (-27.0, -22.0), 'BMSPA': (-16.5, -12.5), 'MMSPA': (-8.5, -5.0), 'H3PO4': (-1.0, 3.5)}
C_WINDOWS_PPM = {'TMSOH': (0.05, 0.45), 'HMDSO': (0.6, 1.1), 'TMSOEG': (-1.95, -1.45)}
p31 = inventory[(inventory['nucleus'] == '31P') & inventory['sample'].str.contains('TMSPa|H2O')
                & (inventory['temperature_C'] < 25) & (inventory['noe'] == 'FALSE')
                & (inventory['relaxation_delay_s'] >= 5)]
c13 = inventory[(inventory['nucleus'] == '13C') & inventory['sample'].str.startswith('TMSOH')]
shares_P = tabulate_area_shares(p31, P_WINDOWS_PPM, region_ppm=(-32.0, 6.0))
shares_C = tabulate_area_shares(c13, C_WINDOWS_PPM, region_ppm=(-3.0, 2.0))

index = ('sample', 'acquired_at', 'temperature_C', 'hours_since_first')
display(tables.format_share_table(shares_P, list(P_WINDOWS_PPM), index=index))
display(tables.format_share_table(shares_C, list(C_WINDOWS_PPM), index=index))


def measured(shares, sample, which=0):
    """Shares, standard errors and inventory row of one spectrum of a sample (which: position in time order)."""
    rows = shares[shares['sample'] == sample]
    when = sorted(rows['acquired_at'].unique())[which]
    rows = rows[rows['acquired_at'] == when].set_index('window')
    return rows['share'], rows['sigma'], rows.iloc[0]

**Reading.**
- **0.5 % water:** TMSPA still holds 0.93 of the P 19 h after the sample's first spectrum.
- **2 % water:** TMSPA is gone in the first spectrum. H₃PO₄ 0.66–0.67, MMSPA 0.31–0.32 and BMSPA 0.03–0.05 stay the same over the six room-temperature spectra (22–28 June).
- **2023 tubes:** the 16-scan spectra have standard errors of 0.04–0.18, mostly from the baseline. Tube B is the least certain.
- **¹³C:** the TMSOH heated in the probe shows no ring-opening after heating. The one stirred at 80 °C in the glovebox has 0.43 of its Si in TMSOEG and 0.16 in HMDSO.
- **Why ¹H is left out:** in the late spectra of TMSPa + TMSOH (A), TMSOH + H₂O and TMSOTMS + H₂O the solvent appears twice. EC and DEC sit at 4.22 s / 3.81 q / 0.94 t, and the same pattern repeats 0.23 ppm higher (4.45 s / 4.06 q / 1.16 t; notebook 04, B.3). The same molecules are in two magnetic environments, probably sample inside the coaxial liner or a second liquid phase. These are not new products.

### 2.2 From measurements to observations

- **Windows:** measured share ± 2 standard errors, one-sided where the statement is "at least" or "at most".
- **Times:** from the time since the sample's first spectrum (files) to that plus `ASSUMED_MAX_AGE_H`. For the heating in the probe: from the spectra recorded at 80 °C (files) to the whole gap between the room-temperature spectra around them.
- **Concentrations:** Gogoi's recipes. 5 vol% TMSPa = 149 mM; water as in the water series; 2 and 5 vol% TMSOH = 0.18 and 0.45 M, as in the controls.
- **Paper observations** are kept only where no lab spectrum covers them (second table).

In [ ]:
c_05, c_2 = calculate_water_series_c0(0.5), calculate_water_series_c0(2.0)   # 5 vol% TMSPa, 0.5 or 2 vol% water
C_TMSPA_TMSOH = {'TMSPA': c_05['TMSPA'], 'TMSOH': 0.18, 'EC': 7.1}            # recipe of control E4
C_TMSOH = {'TMSOH': 0.45, 'EC': 7.1}                                          # recipe of controls E1–E3
AGE_S = ASSUMED_MAX_AGE_H * 3600.0


def lab_times(row):
    """(t_min, t_max) since mixing [s]: the files give the minimum; the assumed age is added for the maximum."""
    return row['hours_since_first'] * 3600.0, row['hours_since_first'] * 3600.0 + AGE_S


s, e, r = measured(shares_P, '0.5 % H2O')
w = build_share_window(s['TMSPA'], e['TMSPA'], 'at least', K_SIGMA)
t_min, t_max = lab_times(r)
L1 = build_observation('L1', f"0.5 % H2O: TMSPA ≥ {w[0]:.2f} of P after ≥ {t_min / 3600:.0f} h", c0_M=c_05,
                       T_K=r['temperature_C'] + ZERO_CELSIUS_K, t_min_s=t_min, t_max_s=t_max, windows={'tmspa_share': w},
                       status='measured', source='lab ³¹P 2022', time_basis='files + assumed age', probes='hydrolysis')

s, e, r = measured(shares_P, '2 % H2O')
w = build_share_window(s['TMSPA'], e['TMSPA'], 'at most', K_SIGMA)
L2 = build_observation('L2', f"2 % H2O: TMSPA ≤ {w[1]:.2f} of P at the first spectrum", c0_M=c_2,
                       T_K=r['temperature_C'] + ZERO_CELSIUS_K, t_min_s=lab_times(r)[0], t_max_s=lab_times(r)[1],
                       windows={'tmspa_share': w}, status='measured', source='lab ³¹P 2022',
                       time_basis='files + assumed age', probes='hydrolysis')

s, e, r = measured(shares_P, 'TMSPa + TMSOH (A)')
w = build_share_window(s['TMSPA'], e['TMSPA'], 'at most', K_SIGMA)
L3 = build_observation('L3', f"TMSPa + 2 % TMSOH: TMSPA ≤ {w[1]:.2f} of P at the first spectrum", c0_M=C_TMSPA_TMSOH,
                       T_K=r['temperature_C'] + ZERO_CELSIUS_K, t_min_s=lab_times(r)[0], t_max_s=lab_times(r)[1],
                       windows={'tmspa_share': w}, status='measured', source='lab ³¹P 2023',
                       time_basis='files + assumed age', probes='transfer')

probe = inventory[inventory['sample'] == 'TMSOH, probe']
heated = find_heated_windows(probe)
rt = probe[probe['temperature_C'] < 25]
gap_s = (rt.loc[rt['acquired_at'] > heated['end'].max(), 'acquired_at'].min()
         - rt.loc[rt['acquired_at'] < heated['start'].min(), 'acquired_at'].max()).total_seconds()
s, e, r = measured(shares_C, 'TMSOH, probe', which=-1)
w = build_share_window(s['TMSOEG'], e['TMSOEG'], 'at most', K_SIGMA)
L4 = build_observation('L4', f"TMSOH heated in the probe: ring-opened ≤ {w[1]:.2f} of Si", c0_M=C_TMSOH, T_K=353.15,
                       t_min_s=heated['minutes'].sum() * 60.0, t_max_s=gap_s, windows={'ring_opened_fraction': w},
                       status='measured', source='lab ¹³C 2022', time_basis='files (80 °C spectra … gap)',
                       probes='solvent_attack')

s, e, r = measured(shares_C, 'TMSOH, glovebox')
w_eg = build_share_window(s['TMSOEG'], e['TMSOEG'], 'both', K_SIGMA)
w_hm = build_share_window(s['HMDSO'], e['HMDSO'], 'at most', K_SIGMA)
L5 = build_observation('L5', f"TMSOH 8 h at 80 °C: ring-opened {w_eg[0]:.2f}–{w_eg[1]:.2f}, HMDSO ≤ {w_hm[1]:.2f} of Si",
                       c0_M=C_TMSOH, T_K=353.15, t_min_s=8 * 3600.0, t_max_s=8 * 3600.0,
                       windows={'ring_opened_fraction': w_eg, 'hmdso_si_fraction': w_hm}, status='measured',
                       source='lab ¹³C 2022', time_basis='paper (8 h hold), assumed to be this sample',
                       probes=('solvent_attack', 'condensation'))

controls = {obs['id']: obs for obs in observations_from_controls()}
OBSERVATIONS = [L1, L2, L3, L4, L5, controls['E1'], controls['E4']]
LABELS = {obs['id']: f"{obs['id']} · {obs['label'].split(':')[0]}" for obs in OBSERVATIONS}   # short names for figures
display(describe_observations(OBSERVATIONS))

hm_glovebox = measured(shares_C, 'TMSOH, glovebox')
pd.DataFrame([
    ['E1', 'TMSOH in EC, 1 week at RT: no ring-opening', 'used', 'no lab spectrum covers it'],
    ['E2', 'TMSOH, 8 h at 80 °C: TMS-EG clearly formed', 'replaced by L5', 'L5 is the measured ¹³C, likely of the same sample'],
    ['E3', 'TMSOH, 8 h at 80 °C: HMDSO at impurity level (≤ 0.05)', 'replaced by L5',
     f"measured HMDSO share {hm_glovebox[0]['HMDSO']:.2f} ± {hm_glovebox[1]['HMDSO']:.2f}"],
    ['E4', 'TMSPA + TMSOH, RT: TMSPA gone (24 h assumed)', 'used', 'L3 is the lab analogue with measured shares'],
    ['W0.5, W2', 'water series, ³¹P', 'replaced by L1, L2', 'likely the same samples (notebook 04, §1)'],
    ['W1, W5', 'water series, 1 and 5 vol%', 'not used', 'no time is known, not even a lower bound'],
    ['Fig. S2', '2 vol% heated to 80 °C', 'replaced by §5.2', 'the lab spectra of the same ramp'],
    ['tube B', 'TMSPa + TMSOH (B), lab', 'held out', 'kept to test the fit'],
], columns=['id', 'observation', 'use', 'why']).set_index('id')

**Reading.**
- Seven observations remain: five measured in the lab and two from the paper.
- L1 and L4 rest on times from the files. L2 and L3 rest on the assumed age. L5 rests on the paper's 8 h hold, assumed to be this sample.
- Measured windows replace two of the paper's: E2 (5–90 % ring-opened becomes about 34–51 %) and E3 (the assumed ≤ 5 % HMDSO, which the measurement exceeds).

## 3. M0: one barrier for every reaction

One barrier for all nine reactions, scanned from 0.60 to 1.70 eV, in the capped-BEP form and in the Marcus form. Each bar is the range one observation allows.

In [ ]:
M0_MODELS = {'M0 · capped BEP, one E0': get_model('reference_bep'), 'M0 · Marcus, one g': REFERENCE}
m0 = {}
for label, model in M0_MODELS.items():
    intervals = find_allowed_intervals(model, SHARED_BARRIER, OBSERVATIONS)
    m0[label] = (intervals, intersect_intervals(intervals).iloc[0])
fitting_plots.plot_allowed_intervals(m0, LABELS, ncols=2);
pd.concat({label: tables.format_columns(iv.set_index('observation')[['g_low_eV', 'g_high_eV', 'verdict']],
                                        {'g_low_eV': '.3f', 'g_high_eV': '.3f'})
           for label, (iv, _) in m0.items()}, axis=1)

**Reading.** Under both forms, no single barrier satisfies every observation (panel subtitles).
- **Capped BEP:** no value lets TMSPA react with TMSOH at room temperature (L3, E4), because EC takes the TMSOH first.
- **Marcus:** E4 allows no value either. Without E4, transfer still needs g ≤ 1.14 eV (L3), while solvent attack (L4, L5, E1) and the survival of TMSPA at 0.5 % water (L1) need g ≥ 1.27 eV.
- So one barrier cannot serve transfer and solvent attack together. Under the rule, **M0 is rejected**.

## 4. M1: one barrier per family

One Marcus barrier per family. Each family is scanned against the observations that probe it; the other families keep their `level1` values.

In [ ]:
m1 = {}
for family in FAMILIES:
    intervals = find_allowed_intervals(REFERENCE, family, select_observations(OBSERVATIONS, family))
    m1[family] = (intervals, intersect_intervals(intervals).iloc[0])
fitting_plots.plot_allowed_intervals({fitting_plots.FAMILY_TITLES[f]: v for f, v in m1.items()}, LABELS, ncols=2);
m1_region = pd.DataFrame({f: inter for f, (_, inter) in m1.items()}).T
tables.format_columns(m1_region, {'g_low_eV': '.3f', 'g_high_eV': '.3f'})

**Reading.**
- **Transfer** ≤ 1.13 eV (E4) or ≤ 1.14 eV (L3, measured). Both rest on an assumed time.
- **Condensation** ≥ 1.20 eV, from the HMDSO of L5. The measured share is higher than the paper's assumed 5 %, so the bound is looser than before (1.24).
- **Solvent attack** 1.307–1.324 eV, from the measured ring-opening of L5. That range is about seven times narrower than E1 + E2 gave (1.27–1.39), but it rests on the 8 h hold.
- **Hydrolysis is empty.** TMSPA surviving 19 h at 0.5 % water (L1) needs g ≥ 1.30 eV. TMSPA gone at 2 % water within the assumed day (L2) needs g ≤ 1.26 eV. With the assumed age, **M1 is rejected**; §4.2 shows which age would reopen it.

### 4.1 Two barriers at a time

In the water samples hydrolysis and transfer act together: TMSOH released by hydrolysis takes a TMS group from another phosphate. The slice moves both barriers at once.

In [ ]:
# Water samples: edges in hydrolysis traced at each transfer value. L3 and E4 do not involve hydrolysis: their transfer bounds (§4) are horizontal lines.
edges = trace_boundaries(REFERENCE, ('hydrolysis', np.round(np.arange(1.00, 1.601, 0.05), 3)),
                         ('transfer', np.round(np.arange(0.80, 1.401, 0.05), 3)), select_observations(OBSERVATIONS, 'hydrolysis'))
transfer_bounds = m1['transfer'][0].set_index('observation')
fitting_plots.plot_boundary_map(edges, LABELS, title='M1 slice: hydrolysis × transfer (others at level1)', y_bounds={
    LABELS[o]: (None, transfer_bounds.loc[o, 'g_high_eV']) for o in transfer_bounds.index});

**Reading.** The edges of L1 and L2 run almost parallel to the transfer axis and never cross. Transfer cannot reconcile them: faster transfer speeds up TMSPA consumption at both water contents alike. L3 and E4 only cap transfer.

### 4.2 How the bounds move with the assumed times

Three times are assumed: the age of the 2 % sample and of the TMSPa + TMSOH tube at their first spectrum, and the 8 h hold of the glovebox sample. Each is varied in turn while the other observations stay fixed.

In [ ]:
by_id = {obs['id']: obs for obs in OBSERVATIONS}
AGES_D = np.array([0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0])
HOLDS_H = np.array([2.0, 4.0, 8.0, 16.0, 32.0])


def bound(model, family, obs, side):
    return find_allowed_intervals(model, family, [obs])[f'g_{side}_eV'].iloc[0]


g_H_max = [bound(REFERENCE, 'hydrolysis', replace_observation(L2, t_max_s=L2['t_min_s'] + a * 86400.0), 'high') for a in AGES_D]
g_T_max = [bound(REFERENCE, 'transfer', replace_observation(L3, t_max_s=L3['t_min_s'] + a * 86400.0), 'high') for a in AGES_D]
g_SA = [find_allowed_intervals(REFERENCE, 'solvent_attack', [replace_observation(L5, t_min_s=h * 3600.0, t_max_s=h * 3600.0)])
        [['g_low_eV', 'g_high_eV']].iloc[0].to_numpy() for h in HOLDS_H]
g_H_min = m1['hydrolysis'][0].set_index('observation').loc['L1', 'g_low_eV']
m1_sa = m1['solvent_attack'][0].set_index('observation')
fitting_plots.plot_time_scenarios({
    'Hydrolysis: age of the 2 % sample (L2)': {
        'x': AGES_D, 'xlabel': 'assumed time from mixing to the first spectrum (days)',
        'lines': {'L2, TMSPA gone': (g_H_max, 'upper'), 'L1, TMSPA survives': (g_H_min, 'lower')},
        'marks_h': {'assumed in §4': ASSUMED_MAX_AGE_H / 24.0}},
    'Transfer: age of the TMSPa + TMSOH tube (L3)': {
        'x': AGES_D, 'xlabel': 'assumed time from mixing to the first spectrum (days)',
        'lines': {'L3, TMSPA mostly gone': (g_T_max, 'upper')},
        'marks_h': {'assumed in §4': ASSUMED_MAX_AGE_H / 24.0}},
    'Solvent attack: hold of the glovebox sample (L5)': {
        'x': HOLDS_H, 'xlabel': 'assumed time at 80 °C (h)',
        'lines': {'L5, enough ring-opening': ([v[1] for v in g_SA], 'upper'),
                  'L5, not too much ring-opening': ([v[0] for v in g_SA], 'lower'),
                  'E1 and L4, no ring-opening': (max(m1_sa.loc['E1', 'g_low_eV'], m1_sa.loc['L4', 'g_low_eV']), 'lower')},
        'marks_h': {'paper: 8 h': 8.0}},
});
crossing_d = float(np.exp(np.interp(g_H_min, g_H_max, np.log(AGES_D))))
print(f'M1 hydrolysis interval opens if the 2 % sample was at least {crossing_d:.1f} days old at its first spectrum')

**Reading.**
- **Hydrolysis:** M1 survives only if the 2 % sample was mixed at least about 5 days before its first spectrum (printed above). The mixing date decides this. If it was mixed the same day, the water order of M1 (rate ∝ [H₂O]) is wrong, not just the barrier.
- **Transfer:** the upper bound rises by about 0.03 eV per tripling of the age, from 1.09 eV (2.4 h) to 1.26 eV (100 days).
- **Solvent attack:** the interval moves by about 0.02 eV per doubling of the hold. Every hold from 2 to 32 h stays above the E1/L4 lower bound, so the interval is never empty, only shifted.

## 5. Tests that no barrier can pass

### 5.1 The composition path of the 2 % sample

A barrier moves a run along its path but cannot change the path. Here are room-temperature runs with 2 % water for several barrier sets, against the six room-temperature spectra of the 2 % sample. Closest approach: the smallest number of standard errors by which the run misses the first spectrum, counting the worst species.

In [ ]:
s2, e2, r2 = measured(shares_P, '2 % H2O')
T_2 = r2['temperature_C'] + ZERO_CELSIUS_K
target = {sp: s2[sp] for sp in ('BMSPA', 'MMSPA', 'H3PO4')}
target_sigma = {sp: e2[sp] for sp in target}
g_T_bound = m1_region.loc['transfer', 'g_high_eV']


def split_model(g_R1, g_R2, g_R3, g_T):
    return (SPLIT.with_family_params('hydrolysis_R1', g_eV=g_R1).with_family_params('hydrolysis_R2', g_eV=g_R2)
            .with_family_params('hydrolysis_R3', g_eV=g_R3).with_family_params('transfer', g_eV=g_T))


VARIANTS = {   # label → (model, network)
    'M0 · capped BEP, E0 1.15': (get_model('reference_bep'), None),
    'M1 · g_H 1.10': (REFERENCE.with_family_params('hydrolysis', g_eV=1.10), None),
    'M1 · g_H 1.30': (REFERENCE.with_family_params('hydrolysis', g_eV=1.30), None),
    'M1 · g_H 1.50': (REFERENCE.with_family_params('hydrolysis', g_eV=1.50), None),
    f'M1-split · R3 +0.10 eV, g_T {g_T_bound:.2f}': (split_model(1.30, 1.30, 1.40, g_T_bound), SPLIT_NETWORK),
    'M1-split · R3 +0.10 eV, g_T 1.30 (excluded by L3/E4)': (split_model(1.30, 1.30, 1.40, 1.30), SPLIT_NETWORK),
}
paths, approach = {}, {}
for label, (model, network) in VARIANTS.items():
    paths[label] = simulate_phosphate_path(c_2, model, T_K=T_2, t_end_s=3.2e11, network=network)
    best = calculate_path_distance(paths[label], target, target_sigma)
    approach[label] = {'closest approach (SE)': best['distance'], 'at time (days)': best['row']['t_h'] / 24.0,
                       **{f'{sp} there': best['row'][sp] for sp in target}}
ramp = shares_P[shares_P['sample'] == '2 % H2O']
measured_ramp = ramp.pivot_table(index='acquired_at', columns='window', values='share').join(
    ramp.pivot_table(index='acquired_at', columns='window', values='sigma').add_suffix('_sigma'))
fitting_plots.plot_composition_paths(paths, measured_ramp, title='2 % water at room temperature: model paths and the six measured spectra');
pd.DataFrame(approach).T.round(3)

**Reading.**
- Every M0 and M1 path passes H₃PO₄ ≈ 0.6 with BMSPA ≈ 0.15–0.20 and MMSPA ≈ 0.22. The sample has BMSPA 0.05 and MMSPA 0.31. The three M1 paths lie on top of each other: g_H only changes how fast the path is travelled (hours to decades, last table).
- Splitting the hydrolysis steps does not help while transfer stays within its bound. With fast transfer the phosphates exchange TMS groups through TMSOH and HMDSO, so their distribution follows the computed transfer equilibria, not the hydrolysis rates.
- The path reaches the measurement only with transfer slower than L3 and E4 allow.
- **No barrier set within the bounds reproduces the 2 % composition.** It points to the thermodynamics (M3) or to the chemistry of the medium, not to a barrier value.

### 5.2 What the heating of the 2 % sample is worth at room temperature

The files record spectra at 30, 40, …, 80 °C between the room-temperature spectra. Each step lasted at least as long as its spectra, which is the minimum heating. For a step with barrier ΔG‡, that heating counts as a longer time at room temperature (Eyring, ΔS‡ = 0).

In [ ]:
heated_2 = find_heated_windows(inventory[inventory['sample'] == '2 % H2O'])
display(heated_2)
DG_GRID_EV = np.round(np.arange(0.90, 1.401, 0.05), 2)
rt_days = calculate_rt_equivalent_hours(heated_2, DG_GRID_EV, T_ref_K=T_2) / 24.0
span_days = (measured_ramp.index.max() - measured_ramp.index.min()).total_seconds() / 86400.0
change = measured_ramp.iloc[-1] - measured_ramp.iloc[0]
print(f'Change from the first to the last room-temperature spectrum ({span_days:.1f} days apart): '
      + ', '.join(f"{sp} {change[sp]:+.3f} ± {np.hypot(*measured_ramp[f'{sp}_sigma'].iloc[[0, -1]]):.3f}"
                  for sp in P_WINDOWS_PPM))
fitting_plots.plot_rt_equivalent(DG_GRID_EV, rt_days, references={
    f'room-temperature span of the spectra ({span_days:.1f} d)': span_days,
    f'M1 minimum age of the sample ({crossing_d:.1f} d, §4.2)': crossing_d},
    title='Minimum heating of the 2 % sample, as days at room temperature');

**Reading.**
- The sample spent about 17 min at each step, 1.8 h in all, plus 6 days at room temperature between the first and last spectrum.
- For barriers of 1.1–1.3 eV that heating alone is worth about 25–80 days at room temperature. No share moved by more than its standard error.
- Whatever produced the first composition must therefore have taken months, or stopped: an equilibrium (§5.3) or a reactant used up (water, M4).

### 5.3 If the plateau is an equilibrium (M3)

The question: which ΔG_rxn of R2 and R3 would make the plateau an equilibrium? The released TMS groups end as TMSOH or, two at a time, as HMDSO (giving back one water). Each possible split gives one point (`calculate_equilibrium_locus`); measuring TMSOH or water would pick one. The computed values are drawn with their 1 and 2 standard-error ellipses (covariance from the shared species).

In [ ]:
plateau = {sp: measured_ramp[sp].mean() for sp in ('BMSPA', 'MMSPA', 'H3PO4')}
locus = calculate_equilibrium_locus(plateau, tmspa0_M=c_2['TMSPA'], h2o0_M=c_2['H2O'], T_K=T_2)
computed = {r: thermo.loc[r, 'dG_rxn_eV'] for r in ('R2', 'R3')}
covariance = calculate_solvation_covariance(['R2', 'R3'])
closest_eq = find_closest_on_locus(locus, computed, covariance)
fitting_plots.plot_equilibrium_locus(locus, computed, covariance, closest_eq,
                                     title='ΔG_rxn that would make the 2 % plateau an equilibrium');
print(f"closest point: {closest_eq['distance']:.2f} standard errors from the computed values "
      f"(ΔG R2 {closest_eq['row']['dG_R2_eV']:+.3f}, R3 {closest_eq['row']['dG_R3_eV']:+.3f} eV; "
      f"computed {computed['R2']:+.3f}, {computed['R3']:+.3f} eV)")
locus.iloc[::10].round(3)

**Reading.**
- As an equilibrium, the plateau needs ΔG(R3) ≥ −0.01 eV and ΔG(R2) ≥ −0.04 eV.
- The closest such point lies 0.75 standard errors from the computed values (+0.17 eV on R3, +0.12 eV on R2), inside the 1 SE ellipse. **M3 is admissible.**
- If it holds, the plateau is the first measured ΔG_rxn of the network. Measuring TMSOH (²⁹Si) or water (Karl Fischer) in the same tube would pick one point on the locus.

### 5.4 TMSPa without added water (2023)

The TMSPa-alone sample was recorded fresh and 10 days later. The water needed for that conversion follows from stoichiometry alone: one water per TMSPA, or half of one if every released TMSOH transfers to a second TMSPA.

In [ ]:
s_fresh, e_fresh, r_fresh = measured(shares_P, 'TMSPa alone', which=0)
s_late, e_late, r_late = measured(shares_P, 'TMSPa alone', which=-1)
converted_M = (s_fresh['TMSPA'] - s_late['TMSPA']) * c_05['TMSPA']
converted_sigma_M = np.hypot(e_fresh['TMSPA'], e_late['TMSPA']) * c_05['TMSPA']
EC_DEC_DENSITY_G_ML = 0.5 * (1.321 + 0.975)   # EC 1.321, DEC 0.975 g/mL (handbook values), 1:1 by volume
water = calculate_required_water_M(converted_M, density_g_mL=EC_DEC_DENSITY_G_ML)
pd.Series({
    'days between spectra': (r_late['acquired_at'] - r_fresh['acquired_at']).total_seconds() / 86400.0,
    'TMSPA share fresh → late': f"{s_fresh['TMSPA']:.2f} → {s_late['TMSPA']:.2f}",
    'TMSPA converted (mM)': f'{1000 * converted_M:.0f} ± {1000 * converted_sigma_M:.0f}',
    'water needed (mM)': f"{1000 * water['low_M']:.0f}–{1000 * water['high_M']:.0f}",
    'water needed (ppm by mass)': f"{water['low_ppm']:.0f}–{water['high_ppm']:.0f}",
})

**Reading.**
- About 86 ± 11 mM of TMSPA turned into BMSPA in 10 days with no water added. Hydrolysis would need about 670–1350 ppm of water, far above the < 20 ppm typical of battery-grade solvent.
- So either water entered the tube during storage, or TMSPA reacts by a route outside the network.
- Until a Karl Fischer titration of the 2023 solvent decides, this sample is a blank for the 2023 tubes, not a constraint.

## 6. What is left for the first fit

In [ ]:
# M1-split: R1 alone against the water samples, with R2 and R3 at their level1 values (0.80 eV)
split_R1 = find_allowed_intervals(SPLIT, 'hydrolysis_R1', select_observations(OBSERVATIONS, 'hydrolysis'), network=SPLIT_NETWORK)


def closest(prefix, within_bounds=True):
    return min(v['closest approach (SE)'] for k, v in approach.items()
               if k.startswith(prefix) and not (within_bounds and 'excluded' in k))


summary = pd.DataFrame({
    **{label: [summarize_region(iv), 'not needed', 'rejected' if intersect_intervals(iv).iloc[0]['empty'] else 'survives']
       for label, (iv, _) in m0.items()},
    'M1 · g per family': [
        '; '.join(f"{f}: {summarize_region(iv)}" for f, (iv, _) in m1.items()),
        f"misses the 2 % composition by ≥ {closest('M1 ·'):.1f} SE", 'rejected'],
    'M1-split · g per hydrolysis step': [
        f"R1 with R2, R3 at level1 (0.80 eV): {summarize_region(split_R1)}; "
        'TMSOH released by R2 consumes TMSPA through R5, so L1 needs R1 and R2 slow together',
        f"misses by ≥ {closest('M1-split'):.1f} SE while transfer stays within its bound", 'rejected'],
    'M3 · ΔG_rxn of R2, R3 free': ['not scanned', f"plateau as equilibrium: {closest_eq['distance']:.2f} SE from computed",
                                  'admissible: to fit'],
    'M4 · water not available': ['not scanned', 'needs Karl Fischer', 'open'],
}, index=['barrier bounds (§3–4)', 'structure tests (§5)', 'verdict']).T
summary

**Reading.**
- **Barrier-only models fail.** M0 fails on the barrier bounds. M1 and M1-split fail on the composition of the 2 % sample, for any barriers within their bounds and any time since mixing.
- **R1 cannot be bounded alone.** In the water samples, TMSOH released by BMSPA hydrolysis (R2) takes a TMS group from TMSPA (R5), which makes BMSPA again: a loop that consumes TMSPA even when R1 is slow. L1 therefore bounds R1 and R2 together.
- **What survives** is a model in which the thermodynamics of the phosphate ladder can move within its standard error (M3), possibly with water loss (M4).
- **The first fit** should therefore free ΔG_rxn of R2 and R3 (prior: computed ± SE, with their covariance) together with the four family barriers, starting inside the intervals of §4. The hydrolysis interval still depends on the age of the 2 % sample (§4.2). Its data: L1–L5, E1 and E4, plus the plateau as a measured composition. Tube B is held out to test it.

**Answers that would change this region**

| Question | What it changes |
|---|---|
| When was each sample mixed? | the hydrolysis interval (§4.2) and every bound that rests on an assumed age |
| Water content (Karl Fischer) of the 2022 and 2023 solvent, and how the tubes were stored | M4, and whether TMSPa alone is a blank or a new route (§5.4) |
| How long was the glovebox sample stirred at 80 °C? | the solvent-attack interval (§4.2) |
| Was there sample inside the coaxial liner? | whether ¹H can be used (§2.1) |

## 7. Export

Tables of this notebook to `notebooks/results/03/` as CSV (the per-spectrum tables `lab_shares_*.csv` go to the data folder).

In [ ]:
from kinetics.paths import require_data_dir

OUT_DIR = REPO / 'notebooks' / 'results' / '03'
DATA_OUT_DIR = require_data_dir() / 'results' / '03'
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUT_DIR.mkdir(parents=True, exist_ok=True)
exports = {
    'lab_shares_31P.csv': shares_P.drop(columns='path'),
    'lab_shares_13C.csv': shares_C.drop(columns='path'),
    'observations.csv': describe_observations(OBSERVATIONS),
    'm0_intervals.csv': pd.concat({k: iv for k, (iv, _) in m0.items()}),
    'm1_intervals.csv': pd.concat({k: iv for k, (iv, _) in m1.items()}),
    'm1_region.csv': m1_region,
    'edges_hydrolysis_transfer.csv': edges,
    'summary.csv': summary,
    'path_approach.csv': pd.DataFrame(approach).T,
    'equilibrium_locus.csv': locus,
}
for name, df in exports.items():
    in_data_folder = name.startswith('lab_shares_')
    df.to_csv((DATA_OUT_DIR if in_data_folder else OUT_DIR) / name)
    print(f"{'[data folder] ' if in_data_folder else 'notebooks/'}results/03/{name}: {len(df)} rows")